In [ ]:
#https://medium.com/@_init_/how-self-attention-with-relative-position-representations-works-28173b8c245a
#https://arxiv.org/pdf/1803.02155
#https://github.com/AliHaiderAhmad001/Self-Attention-with-Relative-Position-Representations/blob/main/relation_aware_attention.py

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [96]:
def relative_positions(len_q, len_k, k = 2):
    assert 2 * k + 1 <= max(len_q, len_k), "max(len_q, len_k) can't be less than twice of 2 * max_rel_len + 1"
    i = torch.arange(len_q).unsqueeze(1)
    j = torch.arange(len_k).unsqueeze(0)
    distance_matrix = i - j
    distance_matrix = torch.clamp(distance_matrix, -k, k)
    distance_matrix += k
    return distance_matrix.long()

In [97]:
relative_positions(4, 6, 2)

tensor([[2, 1, 0, 0, 0, 0],
        [3, 2, 1, 0, 0, 0],
        [4, 3, 2, 1, 0, 0],
        [4, 4, 3, 2, 1, 0]])

In [98]:
class RelativePosition(nn.Module):
    def __init__(self, max_rel_seq_length, n_embd):
        super().__init__()
        self.k = max_rel_seq_length
        self.n_embd = n_embd
        # for k = 2, [0, 4]
        self.pe = nn.Parameter(torch.empty(2 * max_rel_seq_length + 1, n_embd))
        nn.init.xavier_uniform_(self.pe)

    def forward(self, len_q, len_k):
        embedding = self.pe[relative_positions(len_q = len_q, len_k = len_k, k = self.k)]
        return embedding
        
pe_test = RelativePosition(2, 784)
pe_test(4, 6).shape

torch.Size([4, 6, 784])

In [106]:
n_head: int = 6 # number of heads
n_embd : int = 216 # embedding dimesion
block_size = 8 # max sequence length

In [ ]:
import math
class RelPESelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, k = 2):
        super().__init__()
        self.n_embd = n_embd
        self.n_head = n_head
        self.max_len = k
        assert  n_embd % n_head == 0
        self.head_size = n_embd // n_head
        self.rel_k = RelativePosition(max_rel_seq_length=k, n_embd=n_embd)
        self.rel_v = RelativePosition(max_rel_seq_length=k, n_embd=n_embd)

        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.rpe = RelativePosition(k, self.head_size)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, -1) #q, k, v = qkv.chunk(3, dim=-1)
        #[difference b/w casual self att and with rel if we key & query to be of shape (B, head_size, T, n_heads) rather (B, n_heads, T, head_size]
        r_k1 = k.view(B, T, self.n_head, C // self.n_head).permute(0, 3, 1, 2) # (B, hs, T, nh)
        r_q1 = q.view(B, T, self.n_head, C // self.n_head).permute(0, 3, 1, 2) # (B, hs, T, nh)
        scores = (r_q1 @ r_k1.transpose(-2, -1)) / (1.0 / math.sqrt(self.head_size)) #(B, hs, T, nh) @ B, hs, nh, T) => (B, hs, T, T)
        v = q.view(B, T, self.n_head, C // self.n_head).permute(0, 3, 1, 2) # (B, hs, T, nh)


        # calculate relative position indices and relative bias
        rel_bias = self.rpe(T, T) # (T, T, head_size) (4, 6, 64)
        rel_bias = rel_bias.permute(2, 0, 1).unsqueeze(0) #(1, head_size, T, T) (1, 64, 4, 6)
        rel_bias = rel_bias.view(-1, self.head_size, T, T) #(1, head_size, T, T) 

        #add relative bias to attention scores
        att = scores + rel_bias
        att = att.masked_fill(self.mask[:,:,:T, :T] == 0, float('inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v #(B, hs, T, T) @ (B, hs, T, nh) => (B, hs, T, nh)

        # Concatenate heads and project back
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re assembling all head output side by side
        # output projection
        y = self.c_proj(y)
        return y
        
        

x = torch.randn((32, block_size, n_embd))
print("x", x.shape)
attn = RelPESelfAttention(n_embd=n_embd, n_head=n_head, k=3)
attn(x).shape



x torch.Size([32, 8, 216])


torch.Size([32, 8, 216])

In [ ]:
import torch

# Example: batch_size=2, num_heads=4, seq_len=10, d_k=64
Q = torch.randn(2, 4, 10, 64)
K = torch.randn(2, 4, 10, 64)

# Calculate attention scores using einsum
# 'bhqd' for Query (batch, heads, sequence_length, d_k)
# 'bhkd' for Key (batch, heads, sequence_length, d_k)
# 'bhqk' for output (batch, heads, query_seq_len, key_seq_len)
attention_scores = torch.einsum('bhqd,bhkd->bhqk', Q, K)

import math
d_k = Q.shape[-1]
scaled_attention_scores = attention_scores / math.sqrt(d_k)

attention_weights = torch.softmax(scaled_attention_scores, dim=-1)

attention_weights.shape
